# SCBE-AETHERMOORE: QLoRA Fine-Tuning on Colab

Fine-tune a language model on game training data generated by the Aethermoor RPG.

**Pipeline:**
1. Game generates SFT pairs (choices, battles, dialogues, tower events)
2. Data passes through SCBE governance gates (L9/L12/L14)
3. Approved pairs exported as JSONL to HuggingFace Hub
4. **This notebook**: QLoRA fine-tune on Colab T4 GPU
5. Push trained adapter back to HuggingFace Hub
6. Game loads updated model for NPC intelligence

**Requirements:**
- Colab with T4 GPU runtime (free tier works)
- `HF_TOKEN` stored in Colab Secrets (Settings > Secrets)

---
USPTO #63/961,403 | SCBE v3.0

## 1. Install Dependencies

In [ ]:
!pip install -q \
    transformers>=4.40.0 \
    peft>=0.10.0 \
    bitsandbytes>=0.43.0 \
    accelerate>=0.30.0 \
    datasets>=2.19.0 \
    trl>=0.8.0 \
    huggingface_hub>=0.23.0

print("Dependencies installed.")

## 1b. Install Headless Display Dependencies

Install Xvfb and pyvirtualdisplay so the Pygame game can run on Colab without a monitor.

In [ ]:
# Install Xvfb virtual framebuffer + pygame-ce for headless game rendering
!apt-get -qq install -y xvfb > /dev/null 2>&1
!pip install -q pyvirtualdisplay pygame-ce Pillow numpy

print("Headless display dependencies installed.")
print("  - Xvfb: virtual framebuffer")
print("  - pyvirtualdisplay: Python Xvfb wrapper")
print("  - pygame-ce: game engine")
print("  - Pillow: frame capture + GIF export")

## 2. Authenticate with HuggingFace

Add your `HF_TOKEN` to **Colab Secrets** (key icon in the left sidebar).

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab Secrets.")
except Exception:
    # Fallback: set manually if not in Colab
    if "HF_TOKEN" not in os.environ:
        os.environ["HF_TOKEN"] = input("Enter your HF_TOKEN: ")

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])
print("Logged in to HuggingFace Hub.")

## 3. Configuration

Adjust these settings to match your setup.

In [ ]:
# === Model ===
BASE_MODEL = "microsoft/phi-2"          # 2.7B params, fits T4 with 4-bit
OUTPUT_REPO = "SCBE-AETHER/leader-model-v1"  # Where to push the trained adapter

# === Dataset ===
DATASET_REPO = "SCBE-AETHER/aethermoor-training-v1"  # HF dataset with JSONL files
USE_HF_DATASET = True  # Set False to upload local files instead (see Cell 10)

# === QLoRA ===
LORA_R = 16              # LoRA rank
LORA_ALPHA = 32          # LoRA alpha (scaling)
LORA_DROPOUT = 0.05      # LoRA dropout
# Phi-2 attention projection layers
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "dense"]

# === Training ===
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 4  # Effective batch = 4 * 4 = 16
MAX_SEQ_LENGTH = 512
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01
LOGGING_STEPS = 10
SAVE_STEPS = 50

print(f"Config: {BASE_MODEL} -> {OUTPUT_REPO}")
print(f"QLoRA: r={LORA_R}, alpha={LORA_ALPHA}, targets={TARGET_MODULES}")
print(f"Training: {NUM_EPOCHS} epochs, lr={LEARNING_RATE}, eff_batch={BATCH_SIZE * GRADIENT_ACCUMULATION}")

## 4. Load & Format Dataset

Loads JSONL training data from HuggingFace Hub and converts `prompt`/`response` pairs into the instruction format.

In [ ]:
import json
import glob
from datasets import Dataset, load_dataset


def format_pair(example):
    """Convert a training pair to instruction-response chat format."""
    prompt = example.get("prompt", "")
    response = example.get("response", "")
    event_type = example.get("event_type", "general")

    # Build instruction with event type context
    text = (
        f"### Instruction\n"
        f"[{event_type.upper()}] {prompt}\n\n"
        f"### Response\n"
        f"{response}"
    )
    return {"text": text}


def load_jsonl_files(file_paths):
    """Load multiple JSONL files into a single list of dicts."""
    records = []
    for fp in file_paths:
        with open(fp, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
    return records


if USE_HF_DATASET:
    # Load from HuggingFace Hub
    try:
        raw_dataset = load_dataset(DATASET_REPO, split="train")
        print(f"Loaded {len(raw_dataset)} pairs from HF Hub: {DATASET_REPO}")
    except Exception as e:
        print(f"Could not load from HF Hub: {e}")
        print("Falling back to local upload mode (run Cell 10 first).")
        raw_dataset = None
else:
    raw_dataset = None
    print("HF dataset disabled. Upload local files via Cell 10.")

# If we have local JSONL files uploaded (from Cell 10)
if raw_dataset is None:
    local_files = sorted(glob.glob("/content/training_data/*.jsonl"))
    if local_files:
        records = load_jsonl_files(local_files)
        raw_dataset = Dataset.from_list(records)
        print(f"Loaded {len(raw_dataset)} pairs from {len(local_files)} local files.")
    else:
        raise FileNotFoundError(
            "No training data found. Either:\n"
            "  1. Set USE_HF_DATASET=True and ensure data is on HF Hub\n"
            "  2. Run Cell 10 to upload local JSONL files"
        )

# Format pairs
formatted = raw_dataset.map(format_pair, remove_columns=raw_dataset.column_names)

# Train/eval split
split = formatted.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"\nDataset ready: {len(train_dataset)} train / {len(eval_dataset)} eval")
print(f"\nSample:\n{train_dataset[0]['text'][:300]}...")

## 5. Load Base Model (4-bit Quantized)

Load Phi-2 in 4-bit NF4 quantization to fit on a T4 GPU (16GB VRAM).

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {BASE_MODEL} in 4-bit...")

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False  # Required for gradient checkpointing

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

print(f"Model loaded: {model.num_parameters() / 1e6:.0f}M params")
print(f"GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 6. Apply LoRA Adapters

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for QLoRA training
model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

trainable, total = model.get_nb_trainable_parameters()
print(f"Trainable: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

## 7. Train with SFTTrainer

Fine-tune using the TRL library's SFTTrainer for supervised fine-tuning.

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="/content/checkpoints",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    fp16=True,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
    report_to="none",
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    group_by_length=True,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    args=training_args,
    max_seq_length=MAX_SEQ_LENGTH,
)

print(f"Starting training: {len(train_dataset)} samples, {NUM_EPOCHS} epochs")
print(f"Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"Total steps: ~{len(train_dataset) * NUM_EPOCHS // (BATCH_SIZE * GRADIENT_ACCUMULATION)}")
print()

trainer.train()

print("\nTraining complete!")

# Save final model
trainer.save_model("/content/final_model")
tokenizer.save_pretrained("/content/final_model")
print("Model saved to /content/final_model")

## 8. Push to HuggingFace Hub

Upload the trained LoRA adapter to your HF model repo.

In [ ]:
from huggingface_hub import HfApi

# Create repo if it doesn't exist
api = HfApi()
try:
    api.create_repo(
        repo_id=OUTPUT_REPO,
        repo_type="model",
        exist_ok=True,
        private=True,
    )
except Exception as e:
    print(f"Repo creation note: {e}")

# Push model + tokenizer
model.push_to_hub(OUTPUT_REPO, private=True)
tokenizer.push_to_hub(OUTPUT_REPO, private=True)

# Push a model card
model_card = f"""---
base_model: {BASE_MODEL}
library_name: peft
license: apache-2.0
tags:
  - scbe-aethermoore
  - qlora
  - game-ai
  - sacred-tongues
---

# SCBE-AETHERMOORE Leader Model

QLoRA fine-tuned adapter for the Aethermoor RPG NPC system.

- **Base model**: `{BASE_MODEL}`
- **Training data**: Player choices, battles, NPC dialogues, tower events
- **Governance**: All data passed through SCBE L9/L12/L14 gates
- **LoRA config**: r={LORA_R}, alpha={LORA_ALPHA}, targets={TARGET_MODULES}
- **Sacred Tongues**: KO, AV, RU, CA, UM, DR

USPTO #63/961,403
"""

api.upload_file(
    path_or_fileobj=model_card.encode(),
    path_in_repo="README.md",
    repo_id=OUTPUT_REPO,
    repo_type="model",
)

print(f"\nModel pushed to: https://huggingface.co/{OUTPUT_REPO}")

## 9. Test Inference

Generate a response using the fine-tuned model.

In [ ]:
# Test inference with the trained model
model.eval()

test_prompts = [
    # Game choice
    "### Instruction\n[CHOICE] Scene: aethermoor_arrival. Player chose: 'Teach me everything.' -- Embrace the Protocol\n\n### Response\n",
    # Battle
    "### Instruction\n[BATTLE] [UM] Battle: Izack uses Shadow Bolt on Crystal Sentinel\n\n### Response\n",
    # Dialogue
    "### Instruction\n[DIALOGUE] [KO] NPC 'Polly' (topic: Sacred Tongues Lore): Player says: What are the Six Sacred Tongues?\n\n### Response\n",
]

print("=" * 60)
print("INFERENCE TEST")
print("=" * 60)

for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Show only the generated part
    generated = response[len(tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=True)):]
    print(f"\n--- Prompt ---\n{prompt.strip()[:100]}...")
    print(f"\n--- Generated ---\n{generated.strip()[:200]}")
    print()

## 10. Upload Local JSONL (Alternative)

Use this cell if your training data is local instead of on HuggingFace Hub.  
Upload `.jsonl` files from `training-data/game_sessions/`.

**Run this BEFORE Cell 4** if `USE_HF_DATASET = False`.

In [ ]:
import os
from google.colab import files

# Create upload directory
os.makedirs("/content/training_data", exist_ok=True)

print("Upload your JSONL files (from training-data/game_sessions/).")
print("You can select multiple files at once.\n")

uploaded = files.upload()

for filename, content in uploaded.items():
    dest = f"/content/training_data/{filename}"
    with open(dest, "wb") as f:
        f.write(content)
    # Count lines
    line_count = content.decode("utf-8").count("\n")
    print(f"  Saved: {filename} ({line_count} pairs)")

total_files = len(os.listdir("/content/training_data"))
print(f"\n{total_files} file(s) ready in /content/training_data/")
print("Now run Cell 4 to load and format the data.")

## 11. Training Stats & Cleanup

In [ ]:
import gc

# Training summary
print("=" * 60)
print("TRAINING SUMMARY")
print("=" * 60)
print(f"Base model:      {BASE_MODEL}")
print(f"Output repo:     {OUTPUT_REPO}")
print(f"Training pairs:  {len(train_dataset)}")
print(f"Eval pairs:      {len(eval_dataset)}")
print(f"LoRA rank:       {LORA_R}")
print(f"LoRA alpha:      {LORA_ALPHA}")
print(f"Epochs:          {NUM_EPOCHS}")
print(f"Learning rate:   {LEARNING_RATE}")
print(f"GPU memory used: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
print()
print("Next steps:")
print("  1. Set GOOGLE_AI_KEY in .env for live NPC responses")
print("  2. Play the game to generate more training data")
print("  3. Re-run this notebook periodically to improve the model")
print(f"  4. Model available at: https://huggingface.co/{OUTPUT_REPO}")

# Free GPU memory
del model, trainer
gc.collect()
torch.cuda.empty_cache()
print("\nGPU memory freed.")

## 12. Run Game Headless on Colab (Generate Training Data)

Clone the repo and run the Aethermoor game **headless** on Colab to generate fresh training data.
The AI autopilot plays the game, and you see frames rendered inline. JSONL data is saved for the next training run.

In [ ]:
# Clone the repo (or upload the demo/ folder)
import os

REPO_URL = "https://github.com/SCBE-AETHER/SCBE-AETHERMOORE.git"
REPO_DIR = "/content/SCBE-AETHERMOORE"

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
    print(f"Cloned to {REPO_DIR}")
else:
    !cd {REPO_DIR} && git pull --ff-only
    print(f"Updated {REPO_DIR}")

# Verify demo files exist
demo_dir = os.path.join(REPO_DIR, "demo")
assert os.path.exists(os.path.join(demo_dir, "headless.py")), "headless.py not found"
assert os.path.exists(os.path.join(demo_dir, "aethermoor_game.py")), "aethermoor_game.py not found"
print(f"Game files verified in {demo_dir}")

In [ ]:
import sys
sys.path.insert(0, "/content/SCBE-AETHERMOORE/demo")

from headless import HeadlessDisplay, run_headless

# === Headless Config ===
HEADLESS_STEPS = 500   # Number of game ticks (500 ~ 17 seconds at 30fps)
SHOW_EVERY = 60        # Show a frame inline every N steps
SAVE_GIF = True        # Save an animated GIF of the session

print("=" * 60)
print("HEADLESS GAME SESSION")
print("=" * 60)
print(f"Steps: {HEADLESS_STEPS}")
print(f"AI autopilot: enabled")
print(f"Inline preview: every {SHOW_EVERY} frames")
print()

# Run the game headless with AI autopilot
hd = run_headless(
    num_steps=HEADLESS_STEPS,
    ai_pilot=True,
    save_gif_path="/content/headless_session.gif" if SAVE_GIF else None,
    show_every=SHOW_EVERY,
)

# Show status
status = hd.status()
print(f"\nHeadless Status:")
for k, v in status.items():
    print(f"  {k}: {v}")

## 13. Collect & Push Headless Training Data

After running the game headless, collect the generated JSONL files and push them to the HF dataset repo.

In [ ]:
import glob
import json
from pathlib import Path
from huggingface_hub import HfApi

DATASET_REPO = "SCBE-AETHER/aethermoor-training-v1"

# Find JSONL files generated by the headless session
search_paths = [
    "/content/SCBE-AETHERMOORE/demo/training_output/**/*.jsonl",
    "/content/SCBE-AETHERMOORE/training-data/**/*.jsonl",
]

jsonl_files = []
for pattern in search_paths:
    jsonl_files.extend(glob.glob(pattern, recursive=True))

if not jsonl_files:
    print("No JSONL files found from headless session.")
    print("The game may not have generated training data in this short run.")
    print("Try increasing HEADLESS_STEPS to 1000+ for more data.")
else:
    # Count total pairs
    total_pairs = 0
    for fp in jsonl_files:
        with open(fp) as f:
            total_pairs += sum(1 for line in f if line.strip())

    print(f"Found {len(jsonl_files)} JSONL file(s) with {total_pairs} training pairs:")
    for fp in jsonl_files:
        size = Path(fp).stat().st_size
        print(f"  {fp} ({size:,} bytes)")

    # Push to HuggingFace Hub
    api = HfApi()
    try:
        api.create_repo(
            repo_id=DATASET_REPO,
            repo_type="dataset",
            exist_ok=True,
            private=True,
        )
    except Exception as e:
        print(f"Repo note: {e}")

    for fp in jsonl_files:
        filename = Path(fp).name
        api.upload_file(
            path_or_fileobj=fp,
            path_in_repo=f"headless_sessions/{filename}",
            repo_id=DATASET_REPO,
            repo_type="dataset",
        )
        print(f"  Pushed: {filename}")

    print(f"\nData available at: https://huggingface.co/datasets/{DATASET_REPO}")
    print("Re-run the training cells (4-8) to fine-tune on the new data.")

## 14. Smart ROM Session (Pokemon Crystal)

Run a legally owned Pokemon Crystal ROM with the **smart AI agent** that reads game memory
(party, map, badges, battle state) and generates rich conversation training data.

**Requirements:**
- A `.gb`/`.gbc` ROM file you legally own and dumped yourself
- PyBoy (`pip install pyboy`)
- Optional: Tesseract OCR for dialogue extraction

Upload your ROM below, then run the smart session.

In [ ]:
# Install ROM emulation dependencies
!pip install -q pyboy Pillow numpy
!apt-get -qq install -y tesseract-ocr > /dev/null 2>&1
!pip install -q pytesseract

print("ROM emulation dependencies installed.")
print("  - PyBoy: GB/GBC emulator")
print("  - pytesseract: OCR dialogue extraction")
print("  - Pillow: frame capture + GIF export")

In [ ]:
# Upload your legally owned ROM file
import os
from google.colab import files

ROM_DIR = "/content/roms"
os.makedirs(ROM_DIR, exist_ok=True)

print("Upload your legally owned Pokemon Crystal ROM (.gb or .gbc).")
print("This tool does NOT download ROMs — you must provide your own.\n")

uploaded = files.upload()
rom_path = None
for filename in uploaded:
    dest = os.path.join(ROM_DIR, filename)
    with open(dest, "wb") as f:
        f.write(uploaded[filename])
    rom_path = dest
    print(f"  Saved: {dest} ({len(uploaded[filename]):,} bytes)")

if rom_path:
    print(f"\nROM ready: {rom_path}")
else:
    print("No ROM uploaded. Upload a .gb/.gbc file to continue.")

In [ ]:
# Run the smart ROM session with AI agent
import sys
sys.path.insert(0, "/content/SCBE-AETHERMOORE/demo")

from rom_emulator_bridge import RomEmulatorBridge
from pathlib import Path

# === Smart ROM Config ===
ROM_STEPS = 8000         # Emulation ticks (8000 ~ 2-3 min of game time)
SAMPLE_EVERY = 8         # Capture frames for GIF
OCR_EVERY = 20           # Run OCR on dialogue box
MAX_PAIRS = 600          # Cap training pairs
SMART_AGENT = True       # Use memory-reading AI agent (Pokemon Crystal)
GAME = "auto"            # Auto-detect game from ROM header

print("=" * 60)
print("SMART ROM SESSION (Pokemon Crystal)")
print("=" * 60)
print(f"ROM: {rom_path}")
print(f"Steps: {ROM_STEPS}")
print(f"Smart agent: {SMART_AGENT}")
print(f"Max pairs: {MAX_PAIRS}")
print()

bridge = RomEmulatorBridge(
    rom_path=Path(rom_path),
    out_dir=Path("/content/SCBE-AETHERMOORE/training-data/rom_sessions"),
    steps=ROM_STEPS,
    sample_every=SAMPLE_EVERY,
    hold_ticks=3,
    ocr_every=OCR_EVERY,
    max_pairs=MAX_PAIRS,
    gif_path=Path("/content/rom_preview.gif"),
    gif_fps=10,
    gif_scale=0.45,
    gif_max_frames=220,
    seed=1337,
    smart_agent=SMART_AGENT,
    game=GAME,
)

summary = bridge.run()

print(f"\nSession complete!")
print(f"  ROM:            {summary.rom_title} ({summary.rom_system})")
print(f"  Steps:          {summary.steps}")
print(f"  Training pairs: {summary.pairs}")
print(f"  JSONL:          {summary.jsonl_path}")
if summary.gif_path:
    print(f"  GIF:            {summary.gif_path}")
    from IPython.display import Image, display
    display(Image(filename=summary.gif_path))

## 15. Push ROM Session Data to HuggingFace

Upload the generated JSONL from the smart ROM session to the HF dataset repo.
Then re-run training cells (4-8) to fine-tune on the combined data.

In [ ]:
import glob
from pathlib import Path
from huggingface_hub import HfApi

DATASET_REPO = "SCBE-AETHER/aethermoor-training-v1"

# Find ROM session JSONL files
rom_jsonl = glob.glob("/content/SCBE-AETHERMOORE/training-data/rom_sessions/*.jsonl")

if not rom_jsonl:
    print("No ROM session JSONL files found.")
    print("Run the smart ROM session cell first.")
else:
    total_pairs = 0
    for fp in rom_jsonl:
        with open(fp) as f:
            total_pairs += sum(1 for line in f if line.strip())

    print(f"Found {len(rom_jsonl)} file(s) with {total_pairs} training pairs")

    api = HfApi()
    api.create_repo(repo_id=DATASET_REPO, repo_type="dataset", exist_ok=True, private=True)

    for fp in rom_jsonl:
        name = Path(fp).name
        api.upload_file(
            path_or_fileobj=fp,
            path_in_repo=f"rom_sessions/{name}",
            repo_id=DATASET_REPO,
            repo_type="dataset",
        )
        print(f"  Uploaded: {name}")

    print(f"\nData at: https://huggingface.co/datasets/{DATASET_REPO}")
    print("Re-run cells 4-8 to train on the combined Aethermoor + ROM data.")